# Week 34, Day 2: SOLID: S and O

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w034d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def area(shape):
    kind = shape["kind"]
    if kind == "square":
        return shape["side"] ** 2
    if kind == "rect":
        return shape["w"] * shape["h"]
    raise ValueError(f"unknown shape: {kind}")

print([area(s) for s in [{"kind": "square", "side": 3}, {"kind": "rect", "w": 2, "h": 5}]])
try:
    area({"kind": "circle", "r": 1})
except ValueError as err:
    print(err)

In [ ]:
class Payslip:
    def __init__(self, name, hours, rate):
        self.name, self.hours, self.rate = name, hours, rate

    def pay(self):                        # the payroll rules: finance decides these
        basic = min(self.hours, 40) * self.rate
        overtime = max(self.hours - 40, 0) * self.rate * 3 // 2
        return basic + overtime

    def text(self):                       # the layout: whoever designs payslips decides this
        return f"{self.name}: {self.hours}h, £{self.pay() / 100:.2f}"

    def save(self, path):                 # storage: whoever runs the systems decides this
        with open(path, "a") as f:
            f.write(self.text() + "\n")

p = Payslip("Ngozi", 45, 1200)
print(p.pay(), p.text())

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Timesheet:
    name: str
    hours: int
    rate: int

def pay(t: Timesheet) -> int:
    basic = min(t.hours, 40) * t.rate
    overtime = max(t.hours - 40, 0) * t.rate * 3 // 2
    return basic + overtime

def payslip_text(t: Timesheet, amount: int) -> str:
    return f"{t.name}: {t.hours}h, £{amount / 100:.2f}"

def save_line(path: str, line: str) -> None:
    with open(path, "a") as f:
        f.write(line + "\n")

def run_payroll(sheets: list[Timesheet], path: str) -> None:
    for t in sheets:
        save_line(path, payslip_text(t, pay(t)))

run_payroll([Timesheet("Ngozi", 45, 1200), Timesheet("Tom", 30, 1500)], "payslips.txt")
print(open("payslips.txt").read(), end="")

In [ ]:
import math

class Square:
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

class Rect:
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h

def total_area(shapes):                 # closed: never needs editing for new shapes
    return sum(s.area() for s in shapes)

class Circle:                           # extension: added later, nothing else changed
    def __init__(self, r):
        self.r = r
    def area(self):
        return math.pi * self.r ** 2

print(round(total_area([Square(3), Rect(2, 5), Circle(1)]), 2))

In [ ]:
import json

def to_csv(rows):
    return "\n".join(",".join(str(v) for v in row) for row in rows)

def to_json(rows):
    return json.dumps(rows)

EXPORTERS = {"csv": to_csv, "json": to_json}

def register(name, func):
    if name in EXPORTERS:
        raise ValueError(f"already have an exporter called {name!r}")
    EXPORTERS[name] = func

def export(rows, fmt):
    try:
        exporter = EXPORTERS[fmt]
    except KeyError:
        raise ValueError(f"unknown format {fmt!r}; choose from {sorted(EXPORTERS)}") from None
    return exporter(rows)

rows = [["rice", 450], ["oil", 900]]
print(export(rows, "csv"))
register("tsv", lambda rows: "\n".join("\t".join(str(v) for v in row) for row in rows))
print(export(rows, "tsv").replace("\t", " | "))

> **Key idea.** SOLID names five design principles (Robert C. Martin; acronym by Michael Feathers). Single responsibility: each class or module has one reason to change, meaning one group whose needs drive it; find responsibilities by asking who would request changes, and split them into cohesive parts with a small coordinator. Open-closed (Bertrand Meyer): add behaviour by adding code, not editing working code. Python's extension points are polymorphism (new classes), dictionaries of functions (new entries), and objects or functions passed in. Add them where things really vary.

### Exercise: Splitting a payslip

Split `Payslip` by responsibility into: a frozen dataclass `Timesheet(name, hours, rate)`; a pure function `pay(sheet)` (the same rules: up to 40 hours at the rate, hours beyond 40 at one and a half times the rate, rounded down); a pure function `payslip_text(sheet, amount)` giving the same text as `Payslip.text`; and delete `Payslip`.

In [ ]:
from dataclasses import dataclass

class Payslip:
    def __init__(self, name, hours, rate):
        self.name, self.hours, self.rate = name, hours, rate

    def pay(self):
        basic = min(self.hours, 40) * self.rate
        overtime = max(self.hours - 40, 0) * self.rate * 3 // 2
        return basic + overtime

    def text(self):
        return f"{self.name}: {self.hours}h, £{self.pay() / 100:.2f}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_split():\n    \"split by responsibility\"\n    import dataclasses\n    T = ns[\"Timesheet\"]\n    assert dataclasses.is_dataclass(T) and \"Payslip\" not in ns, \"replace Payslip\"\n    t = T(\"Ngozi\", 45, 1200)\n    assert pay(t) == 57000 and pay(T(\"Tom\", 30, 1500)) == 45000 and pay(T(\"x\", 41, 1001)) == 41541\n    assert payslip_text(t, 57000) == \"Ngozi: 45h, \u00a3570.00\" and payslip_text(t, 1) == \"Ngozi: 45h, \u00a30.01\"")

### Exercise: A dictionary instead of a chain

Replace the `if`/`elif` chain in `convert` with a dictionary `CONVERSIONS` from a pair of units to a function, and add `register(src, dst, func)` so new conversions can be added without editing `convert` (registering a pair that already exists raises `ValueError`). An unknown pair still raises `ValueError`. Start with the three conversions shown.

In [ ]:
def convert(value, src, dst):
    if (src, dst) == ("km", "miles"):
        return value / 1.609344
    elif (src, dst) == ("miles", "km"):
        return value * 1.609344
    elif (src, dst) == ("c", "f"):
        return value * 9 / 5 + 32
    raise ValueError(f"can't convert {src} to {dst}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dispatch():\n    \"a registry\"\n    assert round(convert(42.195, \"km\", \"miles\"), 2) == 26.22 and convert(100, \"c\", \"f\") == 212 and round(convert(1, \"miles\", \"km\"), 3) == 1.609\n    register(\"f\", \"c\", lambda f: (f - 32) * 5 / 9)\n    assert convert(212, \"f\", \"c\") == 100\n    for f in [lambda: convert(1, \"kg\", \"lb\"), lambda: register(\"c\", \"f\", lambda x: x)]:\n        try:\n            f()\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\n    assert \"elif\" not in source and \"==\" not in source, \"use the dictionary instead of comparisons\"")

### Exercise: Shipping rules you can add to

Write a `ShippingCalculator(rules)`: `rules` is a list of rule objects, each with `applies(order) -> bool` and `cost(order) -> int`. `calculate(order)` returns the cost from the **first** rule that applies, and raises `ValueError` if none does. An order is a dictionary with `"pence"` (the basket's value), `"grams"` and `"country"`. Then write three rules:

- `FreeOver(threshold)`: applies to UK orders worth at least `threshold`; costs 0;
- `UKByWeight()`: applies to every UK order; costs 300, plus 100 for each full or part kilogram above the first;
- `Abroad(flat)`: applies to any order that isn't UK; costs `flat`.

`calculate` must not mention any rule by name, so that new rules need no edits to it.

In [ ]:
import math

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rules():\n    \"rules\"\n    S, F, U, A = ns[\"ShippingCalculator\"], ns[\"FreeOver\"], ns[\"UKByWeight\"], ns[\"Abroad\"]\n    calc = S([F(5000), U(), A(1500)])\n    assert calc.calculate({\"pence\": 6000, \"grams\": 900, \"country\": \"UK\"}) == 0\n    assert calc.calculate({\"pence\": 1000, \"grams\": 900, \"country\": \"UK\"}) == 300\n    assert calc.calculate({\"pence\": 1000, \"grams\": 1001, \"country\": \"UK\"}) == 400 and calc.calculate({\"pence\": 1000, \"grams\": 3000, \"country\": \"UK\"}) == 500\n    assert calc.calculate({\"pence\": 9000, \"grams\": 100, \"country\": \"NG\"}) == 1500\n    class Lagos:\n        def applies(self, order):\n            return order[\"country\"] == \"NG\"\n        def cost(self, order):\n            return 800\n    assert S([Lagos(), A(1500)]).calculate({\"pence\": 1, \"grams\": 1, \"country\": \"NG\"}) == 800\n    try:\n        S([U()]).calculate({\"pence\": 1, \"grams\": 1, \"country\": \"FR\"})\n        assert False, \"no rule should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: A command registry

A small text console reads lines like `add rice 450` or `total`. Write:

- `Console()`, holding a basket (a dictionary from item to pence) and a dictionary of commands from name to function;
- `register(name, func)`: adds a command; each command is called as `func(console, args)` with `args` the list of words after the name, and returns a string;
- `run(line)`: splits the line, looks up the command, and returns its result, or `"unknown command: xyz"`;
- three built-in commands, registered in `__init__`: `add item pence` (returns `"added rice"`), `remove item` (returns `"removed rice"` or `"no rice"`), and `total` (returns `"total 1350"`).

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_console():\n    \"commands\"\n    C = ns[\"Console\"]\n    c = C()\n    assert c.run(\"add rice 450\") == \"added rice\" and c.run(\"add oil 900\") == \"added oil\" and c.run(\"total\") == \"total 1350\"\n    assert c.run(\"remove rice\") == \"removed rice\" and c.run(\"remove rice\") == \"no rice\" and c.run(\"total\") == \"total 900\"\n    assert c.run(\"dance\") == \"unknown command: dance\"\n    c.register(\"count\", lambda console, args: f\"{len(console.basket)} items\")\n    assert c.run(\"count\") == \"1 items\" and C().run(\"count\") == \"unknown command: count\", \"each console has its own commands\"")

### Bug hunt: Registered under the wrong name

`export(rows, "csv")` fails with `ValueError: unknown format 'csv'`, though `csv` was registered. Fix `register` so the program prints `rice,450`.

In [ ]:
EXPORTERS = {}

def register(name, func):
    EXPORTERS[func.__name__] = func

def export(rows, fmt):
    if fmt not in EXPORTERS:
        raise ValueError(f"unknown format {fmt!r}")
    return EXPORTERS[fmt](rows)

def comma_separated(rows):
    return "\n".join(",".join(str(v) for v in row) for row in rows)

register("csv", comma_separated)
print(export([["rice", 450]], "csv"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"exported\"\n    expect_output(\"rice,450\")")

### Bug hunt: Every rate is twenty

A loop registers a VAT function for each rate, but they all use 20 per cent, so this prints `[200, 200, 200]` instead of `[0, 50, 200]`. A lambda looks up `rate` when it's **called**, by which time the loop has finished. Fix it by giving each lambda its own copy of the rate as a default argument.

In [ ]:
VAT = {}
for name, rate in [("zero", 0), ("reduced", 5), ("standard", 20)]:
    VAT[name] = lambda pence: pence * rate // 100

print([VAT[n](1000) for n in ["zero", "reduced", "standard"]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"each rate\"\n    expect_output(\"[0, 50, 200]\")")

### Bug hunt: The new payment that adds nothing

A gift card class was added to the payment family, but adding up the fees crashes with `TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'`. Its author forgot part of the interface. Fix `GiftCard` (a gift card has no fee) so the program prints `35`.

In [ ]:
class Payment:
    def __init__(self, pence):
        self.pence = pence

    def fee(self):
        pass

class Card(Payment):
    def fee(self):
        return self.pence * 15 // 1000

class Transfer(Payment):
    def fee(self):
        return 20

class GiftCard(Payment):
    def redeem_code(self):
        return "GC-" + str(self.pence)

print(sum(p.fee() for p in [Card(1000), Transfer(500), GiftCard(2000)]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the fees\"\n    expect_output(\"35\")")

### Puzzle: Adding a format without editing

Arrange the lines so that a new `upper` formatter is added to the registry, without changing `show`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Responsibilities and extensions

Answer each question with a number or `True`/`False`.

In [ ]:
# p: How many reasons to change does this class have (who would ask for changes)?
#    class Invoice: calculate_tax(), render_pdf(), email_to_customer()
# q: True or False: adding a new shape to a dictionary of area functions needs total_area() edited.
# r: True or False: SOLID principles should be applied to every class, however small.
# s: How many classes must change to add a "bank transfer" payment, if payments are polymorphic
#    and the checkout just calls payment.fee()? (Count the new class.)
# t: True or False: a coordinator function that calls four single-purpose functions breaks the
#    single responsibility principle.

p = ___
q = ___
r = ___
s = ___
t = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (3, False, False), \"p to r\"\n    assert (s, t) == (1, False)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a pluggable sales report

Build a sales report designed for both principles: each part has one responsibility, and new groupings and formats can be added without editing anything.

- `Sale`, a frozen dataclass: `day: str`, `shop: str`, `item: str`, `pence: int`.
- `parse_sales(text)`: CSV text with the header `day,shop,item,pence`, into a list of `Sale`; a bad line raises `ValueError`.
- `GROUPINGS`: a dictionary from a grouping's name to a function from a `Sale` to its group. Start with `"shop"`, `"day"` and `"item"`.
- `group_totals(sales, by)`: a dictionary from group to total pence, using `GROUPINGS[by]`, with the groups sorted; an unknown grouping raises `ValueError`.
- `FORMATS`: a dictionary from a format's name to a function from that dictionary of totals to text. Start with `"text"` (lines like `Ikeja: £31.00`) and `"csv"` (a header line `group,pence`, then lines like `Ikeja,3100`).
- `report(sales, by, fmt)`: the totals in that format; an unknown format raises `ValueError`.

Adding a grouping or format must only need a new entry in the dictionary.

In [ ]:
from dataclasses import dataclass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("DATA = \"\"\"day,shop,item,pence\n2026-11-02,Ikeja,suya,1500\n2026-11-02,Yaba,zobo,1500\n2026-11-03,Ikeja,zobo,600\n2026-11-03,Ikeja,suya,1000\n\"\"\"\ndef test_parts():\n    \"the parts\"\n    S = ns[\"Sale\"]\n    sales = parse_sales(DATA)\n    assert sales[0] == S(\"2026-11-02\", \"Ikeja\", \"suya\", 1500) and len(sales) == 4\n    assert group_totals(sales, \"shop\") == {\"Ikeja\": 3100, \"Yaba\": 1500} and list(group_totals(sales, \"day\")) == [\"2026-11-02\", \"2026-11-03\"]\n    assert group_totals(sales, \"item\") == {\"suya\": 2500, \"zobo\": 2100}\n    assert report(sales, \"shop\", \"text\") == \"Ikeja: \u00a331.00\\nYaba: \u00a315.00\"\n    assert report(sales, \"item\", \"csv\") == \"group,pence\\nsuya,2500\\nzobo,2100\"\n    for f in [lambda: report(sales, \"colour\", \"text\"), lambda: report(sales, \"shop\", \"pdf\"), lambda: parse_sales(\"day,shop,item,pence\\na,b,c\\n\")]:\n        try:\n            f()\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\ndef test_extend():\n    \"extending without edits\"\n    sales = parse_sales(DATA)\n    GROUPINGS[\"shop_item\"] = lambda s: f\"{s.shop}/{s.item}\"\n    FORMATS[\"markdown\"] = lambda totals: \"\\n\".join(f\"| {g} | {p} |\" for g, p in totals.items())\n    assert report(sales, \"shop_item\", \"markdown\") == \"| Ikeja/suya | 2500 |\\n| Ikeja/zobo | 600 |\\n| Yaba/zobo | 1500 |\"")

## Warm-up

`area` works, but every new shape means opening it up and editing it, along with every other function that has the same `if kind == ...` chain (`perimeter`, `draw`, `describe`). Each edit risks breaking a shape that already worked. Today's second principle is about designing so that new kinds of thing are **added**, not edited in.

## SOLID

**SOLID** is an acronym for five principles of object-oriented design, collected by Robert C. Martin around 2000 and given the acronym by Michael Feathers:

| Letter | Principle | In short |
|---|---|---|
| S | Single responsibility | a class has one reason to change |
| O | Open-closed | add behaviour by adding code, not editing working code |
| L | Liskov substitution | a child must work wherever its parent does |
| I | Interface segregation | depend on small interfaces, not big ones |
| D | Dependency inversion | depend on abstractions, and have dependencies passed in |

They're guidelines, not laws, and like all design advice they can be overdone. Used with judgement, they describe what most well-designed Python code already does. Today is S and O; tomorrow is L, I and D.

## Single responsibility

The **single responsibility principle** says a class (or module, or function) should have **one reason to change**. Robert Martin later sharpened it: "responsible to one actor", meaning one group of people whose needs drive changes to it.

Here's a class with three reasons to change:

If finance changes the overtime rule, if the payslip layout changes, or if payslips move from a file to a database, this one class changes each time, and every change risks the other two jobs. Splitting it by responsibility gives three small parts, each changed for one reason, and a simple coordinator that connects them:

The pay rules can now be tested with nothing but numbers, the layout can change without touching pay, and the storage can change without touching either. The coordinator, `run_payroll`, has a responsibility too ("run the payroll in this order"), which is fine: it's small, and it changes only when the process changes.

## Finding responsibilities

How do you tell how many responsibilities a class has? Some practical tests:

- **Ask who would request a change.** Each different answer (finance, the people who design the layout, the people who run the servers) is a responsibility.
- **Describe it in one sentence.** "It calculates pay **and** formats payslips **and** saves them" lists three.
- **Look at the imports.** A module that imports `json`, `smtplib` and `statistics` probably saves, emails and calculates.
- **Look at which attributes each method uses.** If the methods fall into groups that use different attributes, each group may want to be its own class.

Don't take it to extremes. A class with one method per class is not the goal; each part should be a **meaningful** unit that you'd describe as one job. A `Matrix` with twenty methods all about matrices has one responsibility.

## Open for extension, closed for modification

The **open-closed principle** (from Bertrand Meyer, 1988) says software should be **open for extension** (you can add new behaviour) but **closed for modification** (you add it without editing code that already works and is tested).

Why does that matter? Code that works has been tested, reviewed and relied upon. Every edit to it can break something that used to work, which is why the warm-up's `if kind == ...` chains are fragile. A design that's open-closed has **extension points**: places where new behaviour plugs in. The warm-up rewritten with Week 30's polymorphism:

## Three ways to stay open

Python offers several kinds of extension point. Choose the lightest one that fits:

| Tool | New behaviour is added by... | Good for |
|---|---|---|
| polymorphism (Week 30) | writing a new class with the same methods | kinds of thing with several behaviours each (area, perimeter, draw) |
| a dictionary of functions | adding an entry: a name and a function | choosing one operation by name: formats, commands, units |
| objects passed in (Week 30, Day 5) | passing a different object or function | behaviour chosen by whoever calls: pricing rules, sort orders, outputs |

A **dictionary of functions** (often called a registry) replaces an `if`/`elif` chain on a name:

Adding `tsv` didn't touch `export`. Week 22's command-line programs (with subcommands) and many plugin systems work exactly like this. (`from None` after `raise` hides the original `KeyError` in the traceback, since it adds nothing for the user.)

Be honest about **where** things vary. Extension points cost something: more names to read and more indirection. Add one when there's a real second case, or a clearly expected one, not for every imaginable future (Day 4's YAGNI). A function with an `if` for two cases that will never grow is fine.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick one `if`/`elif` chain in your own code that chooses behaviour by a name or a kind, and decide which of today's three extension points would replace it best, or whether it's simple enough to leave. Tomorrow: the rest of SOLID, Liskov substitution, interface segregation and dependency inversion.